# Parameter Optimization and Statistical analysis

In [1]:
from bose_harmonic import (
  wavefunction,
  wavefunction_derivative,
  local_energy,
  drift_force,
  HarmonicParams,
)
from vmc import Metropolis
from stats import timeseries_bootstrap
import numpy as np

cycles = 10_000
time_step = 0.05
diffusion_coefficient = 0.5
learning_rate = 0.01
optimization_iterations = 100

number_particles = 100
dim = 3

simulation = Metropolis[HarmonicParams](number_particles, dim)

parameter_guess = HarmonicParams(alpha=0.7)

energy, parameters = simulation.optimize(
  wavefunction,
  wavefunction_derivative,
  local_energy,
  drift_force,
  parameter_guess,
  time_step,
  diffusion_coefficient,
  cycles,
  optimization_iterations,
  learning_rate,
)


cycles_sample = 2**20
_, energies = simulation.sample_importance(
  wavefunction,
  local_energy,
  drift_force,
  parameters,
  time_step,
  diffusion_coefficient,
  cycles_sample,
)

print(parameters)

samples = 2**12
block_size = 2**10

result = timeseries_bootstrap(energies, np.mean, samples, block_size)
print(result)

iteration 10/100: E=153.03, grad=5.167e+01, HarmonicParams(alpha=np.float64(0.5999999999999999))
iteration 20/100: E=150.03, grad=6.069e+00, HarmonicParams(alpha=np.float64(0.4999999999999998))
Finished at 20 iterations
Energy=150.00, grad=8.504e-11, HarmonicParams(alpha=np.float64(0.4999999999999998))
HarmonicParams(alpha=np.float64(0.4999999999999998))
BootstrapResult(statistic=150.0, bias=0.0, standard_error=0.0)


In [ ]:
import jax.numpy as jnp
import numpy as np
import optax

from bose_harmonic_jax import (
  wavefunction_jax,
  wavefunction_derivative_jax,
  local_energy_jax,
  drift_force_jax,
  HarmonicParams,
)
from vmc_jax import MetropolisJAX

cycles = 10_000
time_step = 0.001
diffusion_coefficient = 0.5
learning_rate = 0.01
optimization_iterations = 100

number_particles = 100
dim = 3

simulation = MetropolisJAX[HarmonicParams](number_particles, dim)

parameter_guess = HarmonicParams(alpha=jnp.array(0.7))

optimizer = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(learning_rate))

energy, parameters = simulation.optimize(
  wavefunction_jax,
  wavefunction_derivative_jax,
  local_energy_jax,
  drift_force_jax,
  parameter_guess,
  time_step,
  diffusion_coefficient,
  cycles,
  optimization_iterations,
  optimizer,
)

cycles_sample = 2**20

_, energies = simulation.sample_importance(
  wavefunction_jax,
  local_energy_jax,
  drift_force_jax,
  parameters,
  time_step,
  diffusion_coefficient,
  cycles_sample,
)

print(parameters)

samples = 2**12
block_size = 2**10

result = timeseries_bootstrap(np.array(energies), np.mean, samples, block_size)
print(result)


iteration 0/100: E=184.69, grad=5.057e+01, HarmonicParams(alpha=Array(0.69000006, dtype=float32))
iteration 10/100: E=166.75, grad=3.811e+01, HarmonicParams(alpha=Array(0.59000075, dtype=float32))
iteration 20/100: E=150.00, grad=7.519e-02, HarmonicParams(alpha=Array(0.4909906, dtype=float32))
iteration 30/100: E=147.20, grad=1.197e+01, HarmonicParams(alpha=Array(0.4852791, dtype=float32))
iteration 40/100: E=150.98, grad=3.250e+00, HarmonicParams(alpha=Array(0.50360924, dtype=float32))
iteration 50/100: E=150.13, grad=9.019e-01, HarmonicParams(alpha=Array(0.50040686, dtype=float32))
iteration 60/100: E=149.93, grad=6.905e-04, HarmonicParams(alpha=Array(0.49920425, dtype=float32))
Finished at iteration 62: E=149.93, grad=6.905e-04, HarmonicParams(alpha=Array(0.49920425, dtype=float32))
HarmonicParams(alpha=Array(0.49920425, dtype=float32))
